# This is an example of coding in Python I conducted as a freelance project for a delivery firm. It takes raw data on courier deliveriers, calculates salaries for each courier and returns the result in an excel file

In [1]:
import pandas as pd
import datetime as dt
import numpy as np
import warnings
import re 
import os

from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Border, Side, Alignment, Protection, Font, Color 
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils.cell import get_column_letter, quote_sheetname
from openpyxl.formatting import Rule
from openpyxl.formatting.rule import ColorScaleRule, CellIsRule, FormulaRule
from openpyxl.styles.differential import DifferentialStyle




pd.set_option('display.max_rows', 2000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_colwidth', 10000)

date_format = '%d.%m.%Y'
nice_sep = "_-* # ##0.00_-;-* # ##0.00_-;_-* \"-\"??_-;_-@_-"
thin = Side(border_style = "thin", color = '000000')
the_border = Border(top = thin, left = thin, bottom = thin, right = thin)
bad_fill = PatternFill(bgColor = 'FFC7CE')
good_fill = PatternFill(bgColor = 'C6EFCE')
blank_fill = PatternFill(bgColor = 'FFFFFF')

In [20]:
payment_method = {
    
    'Гаранина Виктория'   : 'НЕТ'        ,
    'Давыдов Юрий'        : 'Н'          ,
    'Жмаев Владимир'      : 'ИП КЕЛЛЕР'  ,
    'Жмаев Андрей'        : 'ИП КЕЛЛЕР'  ,
    'Кравченко Владислав' : 'НЕТ'        ,
    'Кравченко Марк'      : "НЕТ"        ,
    'Ксензенко Александр' : "НЕТ"        ,
    'Махмудов Сергей'     : "НЕТ"        ,  
    'Понамарев Евгений'   : "ЧЕК"        ,
    'Сидоренко Владимир'  : "ЧЕК"        ,
    'Степкин Сергей'      : "ИП КЕЛЛЕР"  ,
    'Фикс Алла'           : "СД"         ,
    'Фикс Вячеслав'       : "СД"         ,
    "Шакуров Станислав"   : "ИП КЕЛЛЕР"  ,
    "Шуманский Алексей"   : "ИП"         ,
    'Бочаров Александр'   : 'НЕТ'        , 
    "Кабаров Сарвар"      : "СД"         ,
    "Касенов Марат"       : "СД"         ,  
    "Николаев Андрей"     : "НЕТ"        ,
    "Норов Зиедулло"      : "ЧЕК"        ,
    "Осколков Константин" : "НЕТ"        ,
    "Чанбаев Дамир"       : "СД"         ,
    "Волошин Павел"       : "ЧЕК"        ,
    "Гедревич Алексей"    : "ЧЕК"        ,
    "Гурьянов Александр"  : "НЕТ"        ,
    "Косумов Зелимхан"    : "НЕТ"        ,
    "Мартыненко Юрий"     : "ЧЕК"        ,
    "Небогин Сергей"      : "ЧЕК"        ,
    "Перескоков Антон"    : "ЧЕК"        ,
    "Попов Игорь"         : "НЕТ"        ,
    "Щербаков Сергей"     : "ИП"         ,
    "Бабкин Александр"    : "Н"          ,
    "Сергеев Игорь"       : "НЕТ"        ,  
    "Сидоренко Анна"      : "НЕТ"        ,
    "Сулейманова Эльвира" : "НЕТ"        ,
    "Даудов Курбан"       : "НЕТ"        ,
    "Третьяков Максим"    : "НЕТ"        ,
    'Ведерников Сергей'   : 'ИП'         ,
    'Рева Роман'          : "ЧЕК"        ,
    "Пономаренко Евгений" : "ЧЕК"        ,
    "Лапшина Алена"       : "ЧЕК"        ,
    'Лукин Сергей'        : 'СД'         ,
    'Волукас Сергей'      : 'ИП КАЗАНЦЕВ', 
    'Степкин Сергей'      : 'ИП КЕЛЛЕР'  , 
    'Джумъаев Рашид'      : 'Н'          ,
    'Гайер Ольга'         : 'ЧЕК'        , 
    'Максименко Лидия'    : 'Н'          ,
    'Прасолова Наталья'   : 'ЧЕК'        ,
    'Пшунова Алина'       : 'НЕТ'        ,
    'Усков Сергей'        : 'ИП КЕЛЛЕР'  , 
    'Усков Роман'         : 'ИП КЕЛЛЕР'  , 
    'Михайловский Евгений': 'ИП КЕЛЛЕР'  ,
    'Шемета Александр'    : 'ЧЕК'        ,
    'Пургин Никита'       : 'Н'          ,
    'Свириков Виталий'    : 'ИП КЕЛЛЕР'  , 
    'Скакалин Денис'      : 'ЧЕК'        ,
    'Дюкин Алексей'       : 'ЧЕК'        ,
    'Максименко Даниил'   : 'ИП КЕЛЛЕР'  ,
    'Леонтьев Алексей'    : 'ЧЕК'        ,
    'Алексанян Артур'     : 'СД'         ,
    'Макаров Тарас'       : 'СД'         ,
    'Сидорук Инна'        : 'Н'          ,
    'Савельев Евгений'    : 'ЧЕК'        ,
    'Попов Антон'         : 'НЕТ'        ,
    'Козачук Екатерина'   : 'НЕТ'        ,
    'Гулеватый Сергей'    : 'ЧЕК'        ,
    'Бабаев Санат'        : 'ИП КЕЛЛЕР'  ,
    'Бутлеровская Олеся'  : 'ЧЕК'        ,
    'Хазиахметов Данис'   : 'ЧЕК'        

}

In [21]:
payment_sum = {
    
    'Гаранина Виктория'   : 0  ,
    'Давыдов Юрий'        : 110,
    'Жмаев Владимир'      : 0  ,
    'Жмаев Андрей'        : 0  ,
    'Кравченко Владислав' : 110,
    'Кравченко Марк'      : 0  ,
    'Ксензенко Александр' : 0  ,
    'Махмудов Сергей'     : 0  , 
    'Понамарев Евгений'   : 110,
    'Сидоренко Владимир'  : 110,
    'Степкин Сергей'      : 0  ,
    'Фикс Алла'           : 110,
    'Фикс Вячеслав'       : 190,
    "Шакуров Станислав"   : 0  ,
    "Шуманский Алексей"   : 110,
    'Бочаров Александр'   : 0  , 
    "Кабаров Сарвар"      : 120,
    "Касенов Марат"       : 120, 
    "Николаев Андрей"     : 0  ,
    "Норов Зиедулло"      : 120,
    "Осколков Константин" : 0  ,
    "Чанбаев Дамир"       : 120,
    "Волошин Павел"       : 110,
    "Гедревич Алексей"    : 110,
    "Гурьянов Александр"  : 0  ,
    "Косумов Зелимхан"    : 0  ,
    "Мартыненко Юрий"     : 110,
    "Небогин Сергей"      : 110,
    "Перескоков Антон"    : 110,
    "Попов Игорь"         : 0  ,
    "Щербаков Сергей"     : 150,
    "Бабкин Александр"    : 120,
    "Сергеев Игорь"       : 0  , 
    "Сидоренко Анна"      : 0  ,
    "Сулейманова Эльвира" : 0  ,
    "Даудов Курбан"       : 0  ,
    "Третьяков Максим"    : 0  ,
    'Ведерников Сергей'   : 110,
    'Рева Роман'          : 110,
    "Пономаренко Евгений" : 110,
    "Лапшина Алена"       : 110,
    'Лукин Сергей'        : 110,
    'Волукас Сергей'      : 0  ,
    'Степкин Сергей'      : 0  , 
    'Джумъаев Рашид'      : 110,
    'Гайер Ольга'         : 110,
    'Максименко Лидия'    : 110,
    'Прасолова Наталья'   : 110,
    'Пшунова Алина'       : 0  ,
    'Усков Сергей'        : 0  , 
    'Усков Роман'         : 0  ,
    'Михайловский Евгений': 0  , 
    'Шемета Александр'    : 110, 
    'Пургин Никита'       : 110, 
    'Свириков Виталий'    : 0  , 
    'Скакалин Денис'      : 110,
    'Дюкин Алексей'       : 110,
    'Максименко Даниил'   : 0  ,
    'Леонтьев Алексей'    : 110,
    'Алексанян Артур'     : 110, 
    'Макаров Тарас'       : 110,
    'Сидорук Инна'        : 110,
    'Савельев Евгений'    : 110,
    'Попов Антон'         : 110, 
    'Козачук Екатерина'   : 0  ,
    'Гулеватый Сергей'    : 110,
    'Бабаев Санат'        : 0  ,
    'Бутлеровская Олеся'  : 110,
    'Хазиахметов Данис'   : 110
}

In [4]:
def data_wrap(d) : 
    
        d = d.loc[d['Статус задания'] == 'Выполнено', :]
        d = d[d['Тип Адреса'].isin(['Д', 'З', 'П', 'ПВЗ'])]
        d = d[['Время выполнения', 
               'Адрес', 'Макрозона', 
               '№ документа', 
               'Тип Адреса',
               'Вес, кг', 
               'Дополнительные услуги', 
               'Заказ по заявке'
              ]]
        d = d.rename({'Вес, кг' : 'Расчетный вес, кг'})
        d['Дополнительные услуги'] = d['Дополнительные услуги'].astype(str)
        d['Время выполнения'] = pd.to_datetime(d['Время выполнения'])
        
        foo_cond_1 = d['Дополнительные услуги'].str.contains('Ожидание более 15 мин. у получателя', na = False)
        foo_cond_2 = d['Дополнительные услуги'].str.contains('Ожидание более 15 мин. у отправителя', na = False)
        d['Ожидание'] = foo_cond_1 | foo_cond_2
        
        foo_cond_1 = (d['Дополнительные услуги'].str.contains('Погрузо-разгрузочные работы у получателя', na = False))  & (d['Тип Адреса'] == 'Д')
        foo_cond_2 = (d['Дополнительные услуги'].str.contains('Погрузо-разгрузочные работы у отправителя', na = False)) & (d['Тип Адреса'] == 'З')
        d['ПРР'] = foo_cond_1 | foo_cond_2  
        
        
        foo_cond_1 = (d['Дополнительные услуги'].str.contains('Подъем на этаж лифтом', na = False)) & (d['Тип Адреса'] == 'Д')
        foo_cond_2 = (d['Дополнительные услуги'].str.contains('Подъем на этаж ручной', na = False)) & (d['Тип Адреса'] == 'Д')
        foo_cond_3 = (d['Дополнительные услуги'].str.contains('Спуск с этажа ручной', na = False))  & (d['Тип Адреса'] == 'З')
        foo_cond_4 = (d['Дополнительные услуги'].str.contains('Спуск с этажа лифтом', na = False))  & (d['Тип Адреса'] == 'З')
        d['Подъем'] = foo_cond_1 | foo_cond_2 | foo_cond_3 | foo_cond_4 
        
        foo_cond_1 = (d['Дополнительные услуги'].str.contains('Аренда курьера у получателя', na = False))  & (d['Тип Адреса'] == 'Д')
        foo_cond_2 = (d['Дополнительные услуги'].str.contains('Аренда курьера у отправителя', na = False)) & (d['Тип Адреса'] == 'З')
        d['Аренда курьера'] = foo_cond_1 | foo_cond_2

        d['Ожидание'] = d['Ожидание'].astype(str)
        d['ПРР'] = d['ПРР'].astype(str)
        d['Подъем'] = d['Подъем'].astype(str)
        d['Аренда курьера'] = d['Аренда курьера'].astype(str)
        
        d.loc[d['Ожидание']       == 'True',  'Ожидание']       = 'ДА'
        d.loc[d['ПРР']            == 'True',  'ПРР']            = 'ДА'
        d.loc[d['Подъем']         == 'True',  'Подъем']         = 'ДА'
        d.loc[d['Аренда курьера'] == 'True',  'Аренда курьера'] = 'ДА'
        d.loc[d['Ожидание']       == 'False', 'Ожидание']       = 'НЕТ'
        d.loc[d['ПРР']            == 'False', 'ПРР']            = 'НЕТ'
        d.loc[d['Подъем']         == 'False', 'Подъем']         = 'НЕТ'
        d.loc[d['Аренда курьера'] == 'False', 'Аренда курьера'] = 'НЕТ'
        
        d[['Ожидание Сумма', 'ПРР Сумма', 'Подъем Сумма', 'Аренда сумма', 'КГБ', 'Пригород', 'Промзона', 'Итого']] = np.nan
        d = d.drop(['Дополнительные услуги'], axis = 1)
        
        d = d.sort_values(['Время выполнения'], ascending = True, ignore_index = True)
        
        tt = d[['Время выполнения', 'Адрес', 'Тип Адреса']].copy()
        tt = tt[tt['Тип Адреса'].isin(['ПВЗ', 'П'])]
        tt['Время выполнения'] = tt['Время выполнения'].dt.date
        foo = tt[tt.duplicated()].index
        
        d.loc[foo, 'Брак'] = 1
        d.loc[d['Адрес'] == 'ул. Профсоюзов 64А  ', 'Брак'] = 1 
        d.loc[d['Адрес'] == 'ИНДУСТРИАЛЬНАЯ 38  ', 'Брак'] = 1
        d.loc[d['Адрес'] == 'ул. Индустриальная 38  ', 'Брак'] = 1
        d.loc[d['Адрес'] == '6-й микрорайон 14  ', 'Брак'] = 1
        d.loc[d['Адрес'] == 'ул. Ленина 36 ', 'Брак'] = 1
        d.loc[d['Адрес'] == 'ул. Ленина 2, стр. 4  ', 'Брак'] = 1  
        
        d['order_number'] = d['Заказ по заявке'].combine_first(d['№ документа']).astype(int)
        
        d = d.drop(['order_number'], axis = 1)
        
        return d

In [5]:
def couriers_assemble(path, city) :
    
    count = 0
    
    data = pd.DataFrame()
    
    wb = Workbook()
    ws_sumup = wb.active
    ws_sumup.title = 'Сводка'
  
    for folder in sorted(os.listdir(path)) : 
        
        folder = bytearray(folder.encode())
        folder = folder.replace(b'\xb8\xcc\x86', b'\xb9').decode()
        
        foo = path + '/' + folder 
        
        if os.path.isdir(foo) : 
            
            for file in os.listdir(foo) : 
                
                if file.endswith('.csv') : 
                    
                    count = count + 1 
                    
                    data = pd.concat([data, pd.read_csv(path + '/' + folder + '/' + file, sep = ';')], axis = 0)
                    
        else : 
            
            continue 

        try :
            
            data = data_wrap(d = data)

        except : 

            return folder
        
        wb.create_sheet(folder)
        ws = wb[folder]
        
        for row in dataframe_to_rows(data, index = False, header = True) : 
            
            ws.append(row)
             
        for row in ws.iter_rows() : 
            
            if row[0].row == 1 :  
                
                continue
            
            C2 = row[2].coordinate
            F2 = row[5].coordinate
            H2 = row[7].coordinate
            I2 = row[8].coordinate
            K2 = row[10].coordinate
            L2 = row[11].coordinate
            M2 = row[12].coordinate
            O2 = row[14].coordinate
            P2 = row[15].coordinate
            Q2 = row[16].coordinate
            R2 = row[17].coordinate
            

            row[11].value = f'=IF({H2} = "ДА",95,"")'
            row[12].value = f'=_xlfn.IFS({I2} = "НЕТ","",{F2} < 30,"",{F2} < 50,345,{F2} < 75,575,{F2} < 100,675,{F2} < 150,800,{F2} < 250,1100,{F2} < 350,1625,{F2} < 500,2400,{F2} < 600,2975,{F2} < 700,3600,{F2} < 850,4375,{F2} >= 850,5000)'
            row[14].value = f'=IF({K2} = "ДА",850,"")' 
            row[15].value = f'=IF({F2} >= 20,"ДА","НЕТ")'
            row[16].value = f'=IF({C2} = "ПРИГОРОДЫ","ДА","НЕТ")'
            row[17].value = f'=IF(OR({C2} = "ПРОМЗОНА1",{C2} = "СЕВПРОМЗОН"),"ДА","НЕТ")'

            if (city == 'khms') | (city == 'lan'): 

                the_total = f'=SUM({L2}:{O2})+_xlfn.IFS(AND({P2} = "ДА",{Q2} = "ДА"),200 + 3 * {F2},AND({P2} = "ДА",{R2} = "ДА"),150 + 3 * {F2},{P2} = "ДА",150 + 3 * {F2},{Q2} = "ДА",190,{R2} = "ДА",110,{Q2} = "НЕТ",120)'

            elif (city == 'nzhv') | (city == 'lyan') | (city == 'srg') : 

                the_total = f'=SUM({L2}:{O2})+_xlfn.IFS(AND({P2} = "ДА",{Q2} = "ДА"),200 + 3 * {F2},AND({P2} = "ДА",{R2} = "ДА"),150 + 3 * {F2},{P2} = "ДА",150 + 3 * {F2},{Q2} = "ДА",190,{R2} = "ДА",110,{Q2} = "НЕТ",110)'

            else : 

                return print('Город неправильный') # Переписать
            
            if payment_method[folder] == 'ИП КЕЛЛЕР' :
                
                the_total = f'=SUM({L2}:{O2})+_xlfn.IFS(AND({P2} = "ДА",{Q2} = "ДА"),200 + 3 * {F2},AND({P2} = "ДА",{R2} = "ДА"),150 + 3 * {F2},{P2} = "ДА",150 + 3 * {F2},{Q2} = "ДА",190,{R2} = "ДА",110,{Q2} = "НЕТ",110) + IF(AND({F2}>20,{F2}<30),50,0)'
            
            if row[19].value == 1 : 
                
                row[18].value = 0
                
            else : 
                
                row[18].value = the_total
            
        for column in ws.iter_cols() : 
            
            letter = column[0].column_letter
            
            if letter == 'B' : 
                
                ws.column_dimensions[letter].width = 80
                
            else : 
                
                ws.column_dimensions[letter].width = 22
                
            for cell in column : 
                
                if cell.row == 1 : 
                    
                    cell.font = Font(size = 14, bold = True)
                    cell.alignment = Alignment(horizontal = 'center')
                    cell.border = the_border
                    
                elif (cell.row != 1) & (cell.column == 2) : 
                    
                    cell.font = Font(size = 12)
                    
                elif (cell.row != 1) & (cell.column in [1] + list(range(3, 19))) : 
                    
                    cell.font = Font(size = 12)
                    cell.alignment = Alignment(horizontal = 'center')
                    
                elif (cell.row != 1) & (cell.column == 19) : 
                    
                    cell.font = Font(size = 12)
                    cell.number_format = nice_sep
                    
        if city in ['lyan', 'lan', 'khms'] : 

            ws.column_dimensions['Q'].hidden = True
            ws.column_dimensions['R'].hidden = True
                
        elif city == 'srg' : 
            
            ws.column_dimensions['R'].hidden = True
            
        elif city == 'nzhv' : 
            
            ws.column_dimensions['Q'].hidden = True
            
        ws.column_dimensions['T'].hidden = True
    
        dxf = DifferentialStyle(fill = bad_fill)
        rule = Rule(type = 'containsText', operator = 'containsText', text = 'ДА', dxf=dxf)
        rule.formula = ['NOT(ISERROR(SEARCH("ДА",I2)))']
        ws.conditional_formatting.add('I2:K10000', rule)

    ws_sumup['A1'] = 'Курьер'
    ws_sumup['B1'] = 'Доставок, кол-во'
    ws_sumup['C1'] = 'КГБ'
    ws_sumup['D1'] = 'ПВЗ'
    ws_sumup['E1'] = 'Постамат'
    ws_sumup['F1'] = 'Сумма'
    ws_sumup['G1'] = 'Примечание'
    
    
    names = wb.sheetnames
    names.remove('Сводка')
    
    for i, name in enumerate(names, start = 2) : 
        
        name = quote_sheetname(name)
        ws_sumup.cell(row = i, column = 1).value = name[1:-1]
        ws_sumup.cell(row = i, column = 2).value = f'=COUNT({name}!A:A)'
        ws_sumup.cell(row = i, column = 3).value = f'=COUNTIF({name}!P:P, "ДА")'
        ws_sumup.cell(row = i, column = 4).value = f'=COUNTIF({name}!E:E, "ПВЗ")'
        ws_sumup.cell(row = i, column = 5).value = f'=COUNTIF({name}!E:E, "П")'
        ws_sumup.cell(row = i, column = 6).value = f'=ROUND(SUM({name}!S:S), 0)'
    
    ws_sumup.cell(row = i + 1, column = 1).value = 'Всего'
    ws_sumup.cell(row = i + 1, column = 2).value = '=SUM(B2:' + f'B{i})'
    ws_sumup.cell(row = i + 1, column = 3).value = '=SUM(C2:' + f'C{i})'
    ws_sumup.cell(row = i + 1, column = 4).value = '=SUM(D2:' + f'D{i})'
    ws_sumup.cell(row = i + 1, column = 5).value = '=SUM(E2:' + f'E{i})'
    ws_sumup.cell(row = i + 1, column = 6).value = '=SUM(F2:' + f'F{i})'
    
    
    for column in ws_sumup.iter_cols() : 
        
        letter = column[0].column_letter
        ws_sumup.column_dimensions[letter].width = 21
        
        for cell in column : 
            
            if cell.row == 1 : 
                
                cell.font = Font(size = 14, bold = True)
                cell.alignment = Alignment(horizontal = 'center')
                cell.border = the_border
                
            elif (cell.row == i + 1) & (cell.column != 6): 
                
                cell.font = Font(size = 12, bold = True)
                cell.border = the_border
                cell.alignment = Alignment(horizontal = 'center')
                
            elif (cell.row == i + 1) & (cell.column == 6): 
                
                cell.font = Font(size = 12, bold = True)
                cell.border = the_border
                cell.number_format = nice_sep
                
            elif (cell.row not in [1, i + 1]) & (cell.column == 1) : 
                
                cell.font = Font(size = 12)
                cell.border = the_border
                
            elif (cell.row not in [1, i + 1]) & (cell.column in [2, 3, 4, 5]) :
            
                cell.font = Font(size = 12)
                cell.alignment = Alignment(horizontal = 'center')
                cell.border = the_border
                
            elif (cell.row not in [1, i + 1]) & (cell.column == 6) : 
                
                cell.font = Font(size = 12)
                cell.number_format = nice_sep
                cell.border = the_border
                
        
        
    wb.save(path + '/test.xlsx')
    
    return print(f'Сохранено! Кол-во КК: {count}')

In [8]:
couriers_assemble(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Сургут/Май 2026', 
                  city = 'srg')

Сохранено! Кол-во КК: 256


In [18]:
couriers_assemble(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/НЖВ/Май 2026', 
                  city = 'nzhv')

Сохранено! Кол-во КК: 175


In [22]:
couriers_assemble(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/ХМС/Май 2026', 
                  city = 'khms')
                  
                  

Сохранено! Кол-во КК: 113


In [24]:
couriers_assemble(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Лангепас/Май 2026', 
                  city = 'lan')

Сохранено! Кол-во КК: 24


In [16]:
couriers_assemble(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Лянтор/Май 2026', 
                  city = 'lyan')

Сохранено! Кол-во КК: 28


In [9]:
def final_format(month, wb) : 
    
    ws = wb[month]
    
    ws['A1'] = 'Курьер'
    ws['B1'] = 'Адреса без пост. и ПВЗ, декада 1'
    ws['C1'] = 'Выплата декада 1'
    ws['D1'] = 'Получил (подпись)'
    ws['E1'] = 'Дата'
    ws['F1'] = 'Адреса без пост. и ПВЗ, декада 2'
    ws['G1'] = 'Выплата декада 2'
    ws['H1'] = 'Получил (подпись)'
    ws['I1'] = 'Дата'
    ws['J1'] = 'Итоговый расчет за месяц'
    ws['K1'] = 'Выплата декада 3'
    ws['L1'] = 'Получил (подпись)'
    ws['M1'] = 'Дата'
    ws['N1'] = 'Способ оплаты'

    ws.row_dimensions[1].height = 60

    for column in ws.iter_cols(): 

        letter = column[0].column_letter
        ws.column_dimensions[letter].width = 21

        for cell in column : 

            if cell.row == 1 : 

                cell.font = Font(size = 14, bold = True)
                cell.alignment = Alignment(horizontal = 'center', vertical = 'center', wrapText = True)
                cell.border = the_border

            elif (cell.row != 1) & (cell.column == 1) : 

                cell.font = Font(size = 12)
                cell.border = the_border

            elif (cell.row != 1) & (cell.column in [2, 6, 14]) : 
                
                cell.font = Font(size = 12)
                cell.alignment = Alignment(horizontal = 'center')
                cell.border = the_border

            elif (cell.row != 1) & (cell.column in [3, 7, 10, 11]) : 

                cell.font = Font(size = 12)
                cell.number_format = nice_sep
                cell.border = the_border
                
    
    for letter in ['D', 'E', 'H', 'I', 'L', 'M'] : 
        
        ws.column_dimensions[letter].hidden = True
    
    ws.conditional_formatting.add('K2:K100',

            FormulaRule(formula=['K2 > 0'], stopIfTrue = True, fill = good_fill, font = Font(bold = True)))
    
    ws.conditional_formatting.add('K2:K100',

            FormulaRule(formula=['K2 < 0'], stopIfTrue = True, fill = bad_fill, font = Font(bold = True)))



    return wb 

In [10]:
def final_file(path, month, decade, city, payment_method, payment_sum) :
    
    wb = load_workbook(f'/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Расчет курьеров {city}.xlsx')
    
    if decade == 1 :
        
        wb.create_sheet(month)
        ws = wb[month]
        
    elif decade in [2, 3] : 
        
        ws = wb[month]

    else : 
        
        return print('Неправильно указана декада')  
    
    names = []
    
    for cell in ws['A'] : 
        
        if cell.row != 1 : 
            
            names.append(cell.value)
    
    if decade in [1, 2] :
    
        for folder in sorted(os.listdir(path)) : 
            
            folder = bytearray(folder.encode())
            folder = folder.replace(b'\xb8\xcc\x86', b'\xb9').decode()
            
            data = pd.DataFrame()

            foo = path + '/' + folder 

            if os.path.isdir(foo) : 

                for file in os.listdir(foo) : 

                    if file.endswith('.csv') : 

                        data = pd.concat([data, pd.read_csv(foo + '/' + file, sep = ';', usecols = ['Время выполнения', 'Тип Адреса', 'Статус задания'])], axis = 0)

            else : 

                continue 

            data = data.loc[data['Статус задания'] == 'Выполнено', :]
            data['Время выполнения'] = pd.to_datetime(data['Время выполнения'])

            if   decade == 1 : 

                time_filter = (data['Время выполнения'].dt.day >= 1) & (data['Время выполнения'].dt.day <= 10)

            elif decade == 2 : 

                time_filter = (data['Время выполнения'].dt.day >= 11) & (data['Время выполнения'].dt.day <= 20)
        
            data = data[time_filter]
        
            the_number  = data['Тип Адреса'].value_counts().get('Д', 0) + data['Тип Адреса'].value_counts().get('З', 0)
            
            if folder in names : 
                
                row_num = names.index(folder) + 2
                
                if decade == 1 : 
                    
                    ws.cell(row = row_num, column = 14).value = payment_method[folder]
                    ws.cell(row = row_num, column = 2).value = the_number
                    ws.cell(row = row_num, column = 3).value = f'={ws.cell(row = row_num, column = 2).coordinate} * {payment_sum[folder]}'         
                
                elif decade == 2 : 
                    
                    ws.cell(row = row_num, column = 6).value = the_number
                    ws.cell(row = row_num, column = 7).value = f'={ws.cell(row = row_num, column = 6).coordinate} * {payment_sum[folder]}'
                    
            else : 
                
                ws.cell(row = len(names) + 2, column = 1).value = folder
                ws.cell(row = len(names) + 2, column = 14).value = payment_method[folder]
                
                if decade == 1 : 
                    
                    ws.cell(row = len(names) + 2, column = 2).value = the_number
                    ws.cell(row = len(names) + 2, column = 3).value = f'={ws.cell(row = len(names) + 2, column = 2).coordinate} * {payment_sum[folder]}' 
                    
                elif decade == 2 : 
                    
                    ws.cell(row = len(names) + 2, column = 6).value = the_number
                    ws.cell(row = len(names) + 2, column = 7).value = f'={ws.cell(row = len(names) + 2, column = 6).coordinate} * {payment_sum[folder]}'
                    
                    
                names.append(folder) 
                
        
    
    elif decade == 3 : 
        
        final_dict = pd.read_excel(path + '/' +  f'Итог курьеры {city} {month}.xlsx', 
                                   sheet_name = 'Сводка', 
                                   skipfooter = 1, 
                                   usecols = ['Курьер', 'Сумма'])
        final_dict = final_dict.set_index('Курьер')
        final_dict = final_dict.to_dict()['Сумма']
         
            
        for folder in sorted(os.listdir(path)) : 
            
            folder = bytearray(folder.encode())
            folder = folder.replace(b'\xb8\xcc\x86', b'\xb9').decode()
            
            foo = path + '/' + folder
                
            if (folder not in names) & os.path.isdir(foo):
                    
                    ws.cell(row = len(names) + 2, column = 1).value = folder
                    ws.cell(row = len(names) + 2, column = 14).value = payment_method[folder]
                    names.append(folder)
                    
        for row in ws.iter_rows(): # Важно, что строки. Потому что сначала надо подцепить имя. 

            name = ''

            for cell in row : 

                if (cell.row != 1) & (cell.column == 1) :

                    name = cell.value
                    
                elif (cell.row != 1) & (cell.column == 3) : 
                    
                    C2 = cell.coordinate
                    
                elif (cell.row != 1) & (cell.column == 7) : 
                    
                    G2 = cell.coordinate

                elif (cell.row != 1) & (cell.column == 10) : 

                    if name is None : 

                        pass

                    else : 

                        cell.value = final_dict[name]
                        J2 = cell.coordinate
                    
                elif (cell.row != 1) & (cell.column == 11) : 
                    
                    cell.value = f'= {J2} - {G2} - {C2}'    
                
                
        
        
    wb = final_format(month = month, wb = wb)         
    
    wb.save(f'/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Расчет курьеров {city}.xlsx')
                
    return print(f'Сохранено!')

# Задай параметры! 

In [11]:
decade = 3
month = 'Май 2026'

In [15]:
final_file(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Сургут/' + month, 
           month = month, 
           decade = decade, 
           city = 'Сургут',
           payment_method = payment_method, 
           payment_sum = payment_sum
          )

Сохранено!


In [19]:
final_file(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/НЖВ/' + month, 
           month = month, 
           decade = decade, 
           city = 'НЖВ',
           payment_method = payment_method, 
           payment_sum = payment_sum
          )

Сохранено!


In [23]:
final_file(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/ХМС/' + month, 
           month = month, 
           decade = decade, 
           city = 'ХМС',
           payment_method = payment_method, 
           payment_sum = payment_sum
          )

Сохранено!


In [17]:
final_file(path = '/Users/nvarioshkin/Desktop/CDEK/Курьеры выплаты/Лянтор/' + month, 
           month = month, 
           decade = decade, 
           city = 'Лянтор',
           payment_method = payment_method, 
           payment_sum = payment_sum
          )

Сохранено!
